Load data

In [ ]:
"""
SugarSense — Step 1: PRONOSTIA Dataset Loader & Feature Extractor
=================================================================
Run this first on Google Colab (T4 GPU runtime).

What this does:
  1. Downloads the PRONOSTIA/FEMTO dataset from GitHub
  2. Extracts time-domain & frequency-domain features from vibration signals
  3. Generates health labels (RUL) for each bearing
  4. Saves processed features to /content/sugarsense_features.npz

Time: ~5-10 minutes on Colab
"""

# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "scipy", "numpy",
                "pandas", "matplotlib", "tqdm", "--quiet"], check=True)

# ── Imports ───────────────────────────────────────────────────────────────────
import os, zipfile, requests, io
import numpy as np
import pandas as pd
from scipy import signal
from scipy.stats import kurtosis
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR   = "/content/pronostia"
OUTPUT_NPZ = "/content/sugarsense_features.npz"
WINDOW_SIZE = 2560        # samples per window  (2560 @ 25.6 kHz = 0.1 s)
STEP_SIZE   = 2560        # non-overlapping windows
FS          = 25600       # sampling frequency (Hz) for PRONOSTIA

# PRONOSTIA dataset — GitHub mirror
DATASET_URL = (
    "https://github.com/wkzs111/phm-ieee-2012-data-challenge-dataset/"
    "archive/refs/heads/master.zip"
)

# ── 1. Download & unzip ───────────────────────────────────────────────────────
def download_dataset():
    if os.path.exists(DATA_DIR):
        print(f"[✓] Dataset already present at {DATA_DIR}")
        return
    print("[↓] Downloading PRONOSTIA dataset (~280 MB) …")
    r = requests.get(DATASET_URL, stream=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    buf = io.BytesIO()
    downloaded = 0
    for chunk in r.iter_content(chunk_size=65536):
        buf.write(chunk)
        downloaded += len(chunk)
        print(f"\r    {downloaded/1e6:.1f} / {total/1e6:.1f} MB", end="")
    print()
    print("[✓] Download complete. Extracting …")
    buf.seek(0)
    with zipfile.ZipFile(buf) as zf:
        zf.extractall("/content/")
    # The zip extracts to a folder with -master suffix; rename for convenience
    extracted = "/content/phm-ieee-2012-data-challenge-dataset-master"
    if os.path.exists(extracted):
        os.rename(extracted, DATA_DIR)
    print(f"[✓] Extracted to {DATA_DIR}")

# ── 2. Parse one bearing folder ───────────────────────────────────────────────
def load_bearing_files(bearing_path: str) -> np.ndarray:
    """
    Returns stacked vibration data (N_samples, 2) for one bearing.
    PRONOSTIA CSV format: time, horiz_accel, vert_accel  (no header)
    """
    csv_files = sorted([
        os.path.join(bearing_path, f)
        for f in os.listdir(bearing_path)
        if f.endswith(".csv")
    ])
    if not csv_files:
        return np.array([])
    chunks = []
    for fp in csv_files:
        try:
            df = pd.read_csv(fp, header=None)
            # PRONOSTIA format: time, ch1, ch2, ... accelerations at cols 4,5
            # Some files may have fewer columns — take what we can
            if df.shape[1] >= 6:
                arr = df.iloc[:, 4:6].values.astype(np.float32)
            elif df.shape[1] >= 2:
                arr = df.iloc[:, :2].values.astype(np.float32)
            else:
                continue
            # Ensure exactly 2 columns
            if arr.shape[1] == 1:
                arr = np.hstack([arr, arr])   # duplicate single channel
            chunks.append(arr)
        except Exception:
            continue
    if not chunks:
        return np.array([])
    return np.vstack(chunks)

# ── 3. Feature extraction from one window ────────────────────────────────────
N_FEATURES = 24   # always 12 per channel × 2 channels

def extract_features(window: np.ndarray) -> np.ndarray:
    """
    Extracts 12 features per channel (horizontal + vertical = 24 total).
    Always returns exactly N_FEATURES values, padding with zeros if only
    1 channel is present, or returns None if window is unusable.
    Features:
      Time-domain : RMS, Peak, Crest Factor, Kurtosis, Skewness, Variance
      Freq-domain : FFT band energies × 3 bands, Spectral Entropy, Spectral
                    Centroid, High-freq ratio
    """
    if window.ndim < 2 or window.shape[1] < 1:
        return None
    feats = []
    for ch in range(2):   # always iterate 2 channels
        if ch >= window.shape[1]:
            feats.extend([0.0] * 12)   # pad missing channel with zeros
            continue
        x = window[:, ch]

        # — Time domain —
        rms     = np.sqrt(np.mean(x ** 2))
        peak    = np.max(np.abs(x))
        crest   = peak / (rms + 1e-9)
        kurt    = float(kurtosis(x))
        skew    = float(pd.Series(x).skew())
        var     = float(np.var(x))

        # — Frequency domain —
        freqs   = np.fft.rfftfreq(len(x), d=1.0/FS)
        fft_mag = np.abs(np.fft.rfft(x))

        # Band energies (Hz)
        def band_energy(lo, hi):
            mask = (freqs >= lo) & (freqs < hi)
            return float(np.sum(fft_mag[mask] ** 2))

        b1  = band_energy(0,    1000)       # low
        b2  = band_energy(1000, 5000)       # mid  (bearing defect zone)
        b3  = band_energy(5000, FS//2)      # high

        total_e = b1 + b2 + b3 + 1e-9
        spec_ent = -np.sum(
            (fft_mag/fft_mag.sum() + 1e-12) * np.log(fft_mag/fft_mag.sum() + 1e-12)
        )
        spec_cen = float(np.sum(freqs * fft_mag) / (fft_mag.sum() + 1e-9))
        hf_ratio = b3 / total_e

        feats.extend([rms, peak, crest, kurt, skew, var,
                       b1, b2, b3, spec_ent, spec_cen, hf_ratio])
    return np.array(feats, dtype=np.float32)

# ── 4. Process all bearings ───────────────────────────────────────────────────
def process_all_bearings():
    """
    Walks through all bearing directories, computes features and RUL labels.
    Returns:
        X  — (N_windows, 24)  feature matrix
        y  — (N_windows,)     normalised RUL  0.0 = new → 1.0 = failed
    """
    X_list, y_list = [], []

    # Find all bearing subdirectories across training sets
    bearing_dirs = []
    for root, dirs, files in os.walk(DATA_DIR):
        csv_count = sum(1 for f in files if f.endswith(".csv"))
        if csv_count >= 5:           # a bearing folder has many CSVs
            bearing_dirs.append(root)

    if not bearing_dirs:
        raise RuntimeError(f"No bearing data found in {DATA_DIR}. "
                           "Check the download path.")

    print(f"[→] Found {len(bearing_dirs)} bearing run(s) to process …")

    for bdir in tqdm(bearing_dirs, desc="Bearings"):
        data = load_bearing_files(bdir)
        if data.size == 0 or data.shape[0] < WINDOW_SIZE:
            continue

        total_samples = data.shape[0]
        windows, features = [], []
        for start in range(0, total_samples - WINDOW_SIZE, STEP_SIZE):
            w    = data[start:start + WINDOW_SIZE]
            if w.shape[0] < WINDOW_SIZE:
                continue
            feat = extract_features(w)
            if feat is None or len(feat) != N_FEATURES:
                continue
            features.append(feat)

        if not features:
            continue

        n = len(features)
        # Normalised RUL: 0 = just started, 1 = about to fail
        rul_norm = np.linspace(0.0, 1.0, n, dtype=np.float32)

        X_list.append(np.array(features))
        y_list.append(rul_norm)

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    print(f"[✓] Total windows: {X.shape[0]:,}  |  Features per window: {X.shape[1]}")
    return X, y

# ── 5. Save & quick visualisation ────────────────────────────────────────────
def visualise_sample(X, y):
    fig, axes = plt.subplots(2, 3, figsize=(14, 6))
    fig.suptitle("SugarSense — Feature Distribution Across Bearing Lifecycle",
                 fontsize=13, fontweight="bold")
    feature_names = ["RMS", "Peak", "Crest Factor", "Kurtosis",
                     "FFT Band 2 Energy", "High-Freq Ratio"]
    indices = [0, 1, 2, 3, 7, 11]  # from ch0 features
    for ax, idx, name in zip(axes.flat, indices, feature_names):
        ax.scatter(y, X[:, idx], alpha=0.3, s=4, color="#00b4d8")
        ax.set_xlabel("Normalised RUL (0=new → 1=failed)")
        ax.set_ylabel(name)
        ax.set_title(name)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/feature_distribution.png", dpi=120)
    plt.show()
    print("[✓] Saved feature_distribution.png")

# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    download_dataset()
    X, y = process_all_bearings()
    np.savez_compressed(OUTPUT_NPZ, X=X, y=y)
    print(f"[✓] Features saved → {OUTPUT_NPZ}")
    visualise_sample(X, y)
    print("\n✅  Step 1 complete. Run step2_train_model.py next.")

SHAP version check

In [ ]:
"""
SugarSense — Step 2: Train 1D CNN for Bearing RUL Prediction
=============================================================
Zero external explainability dependencies.
Uses permutation importance instead of SHAP — works on ANY numpy version.

Requirements: tensorflow, scikit-learn, matplotlib, numpy (any version)
"""

# ── Install (only what's needed — no SHAP, no numpy pinning) ─────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + list(args),
                   check=True)

pip("tensorflow", "scikit-learn", "matplotlib")
print("[✓] Packages ready")

# ── Imports ───────────────────────────────────────────────────────────────────
import os, warnings, pickle
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

print(f"[✓] NumPy      {np.__version__}")
print(f"[✓] TensorFlow {tf.__version__}")
print(f"[✓] GPUs       {len(tf.config.list_physical_devices('GPU'))}")

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_NPZ     = "/content/sugarsense_features.npz"
MODEL_DIR     = "/content/sugarsense_model"
os.makedirs(MODEL_DIR, exist_ok=True)

EPOCHS        = 60
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
SEED          = 42
N_FEATURES    = 24

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── 1. Load & validate ────────────────────────────────────────────────────────
print("\n[→] Loading features …")
data = np.load(INPUT_NPZ)
X, y = data["X"].astype(np.float32), data["y"].astype(np.float32)

good = np.isfinite(X).all(axis=1) & np.isfinite(y)
X, y = X[good], y[good]
assert X.shape[1] == N_FEATURES, f"Expected {N_FEATURES} features, got {X.shape[1]}"
print(f"    X: {X.shape}  |  y: {y.shape}")
print(f"    RUL range: {y.min():.3f} → {y.max():.3f}")

# ── 2. Scale ──────────────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)
pickle.dump(scaler, open(f"{MODEL_DIR}/scaler.pkl", "wb"))
print("[✓] Scaler saved")

X_cnn = X_scaled[:, :, np.newaxis]   # (N, 24, 1)

# ── 3. Split ──────────────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cnn, y, test_size=0.2, random_state=SEED, shuffle=True)
print(f"[✓] Train: {X_tr.shape[0]:,}  |  Test: {X_te.shape[0]:,}")

# ── 4. Build 1D CNN ───────────────────────────────────────────────────────────
def build_model(input_shape):
    inp = keras.Input(shape=input_shape, name="vibration_features")

    x = layers.Conv1D(64, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)

    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid", name="rul")(x)

    return keras.Model(inp, out, name="SugarSense_CNN")

model = build_model((N_FEATURES, 1))
model.summary()
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
              loss="mse", metrics=["mae"])

# ── 5. Train ──────────────────────────────────────────────────────────────────
cbs = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True,
                            monitor="val_loss", verbose=1),
    callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6,
                                monitor="val_loss", verbose=1),
    callbacks.ModelCheckpoint(f"{MODEL_DIR}/best_model.keras",
                              save_best_only=True, monitor="val_loss"),
]

print("\n[→] Training …")
history = model.fit(X_tr, y_tr,
                    validation_data=(X_te, y_te),
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    callbacks=cbs,
                    verbose=1)

# ── 6. Evaluate ───────────────────────────────────────────────────────────────
y_pred = model.predict(X_te, verbose=0).flatten()
mae    = mean_absolute_error(y_te, y_pred)
r2     = r2_score(y_te, y_pred)
print(f"\n[✓] Test MAE: {mae:.4f}  |  R²: {r2:.4f}")

# ── 7. Training plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"],     label="Train")
axes[0].plot(history.history["val_loss"], label="Val")
axes[0].set_title("Loss (MSE)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history["mae"],     label="Train")
axes[1].plot(history.history["val_mae"], label="Val")
axes[1].set_title("MAE"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle("SugarSense 1D CNN — Training Curves", fontweight="bold")
plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=120)
plt.show()

plt.figure(figsize=(7, 5))
idx = np.random.choice(len(y_te), min(500, len(y_te)), replace=False)
plt.scatter(y_te[idx], y_pred[idx], alpha=0.4, s=8, color="#00b4d8")
plt.plot([0,1],[0,1], "r--", lw=1.5, label="Perfect")
plt.xlabel("True RUL"); plt.ylabel("Predicted RUL")
plt.title(f"Predicted vs True  (R²={r2:.3f})")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("/content/prediction_scatter.png", dpi=120)
plt.show()
print("[✓] Plots saved")

# ── 8. Permutation Feature Importance (no SHAP, no C extensions) ─────────────
print("\n[→] Computing feature importance via permutation …")

feat_names = (
    [f"H_{n}" for n in ["RMS","Peak","Crest","Kurtosis","Skew","Var",
                         "B1E","B2E","B3E","SpecEnt","SpecCen","HFR"]]
  + [f"V_{n}" for n in ["RMS","Peak","Crest","Kurtosis","Skew","Var",
                         "B1E","B2E","B3E","SpecEnt","SpecCen","HFR"]]
)

# Baseline MAE on a small test subset
N_PERM    = min(500, len(X_te))
X_sub     = X_te[:N_PERM]
y_sub     = y_te[:N_PERM]
base_pred = model.predict(X_sub, verbose=0).flatten()
base_mae  = mean_absolute_error(y_sub, base_pred)

importance = np.zeros(N_FEATURES, dtype=np.float32)
for i in range(N_FEATURES):
    X_perm         = X_sub.copy()
    # Shuffle feature i across all samples (break its relationship with target)
    idx_perm       = np.random.permutation(N_PERM)
    X_perm[:, i, 0] = X_sub[idx_perm, i, 0]
    perm_pred      = model.predict(X_perm, verbose=0).flatten()
    perm_mae       = mean_absolute_error(y_sub, perm_pred)
    importance[i]  = perm_mae - base_mae   # how much worse when feature is shuffled

# Normalise to 0-1
importance = np.clip(importance, 0, None)
if importance.max() > 0:
    importance = importance / importance.max()

sorted_idx = np.argsort(importance)[::-1]

plt.figure(figsize=(12, 5))
bar_colors = ["#ef233c" if i < 5 else "#00b4d8" for i in range(N_FEATURES)]
plt.bar(range(N_FEATURES), importance[sorted_idx],
        color=[bar_colors[i] for i in range(N_FEATURES)])
plt.xticks(range(N_FEATURES),
           [feat_names[i] for i in sorted_idx],
           rotation=45, ha="right", fontsize=8)
plt.ylabel("Permutation Importance (normalised)")
plt.title("SugarSense — Feature Importance | Red = top 5")
plt.grid(alpha=0.2, axis="y"); plt.tight_layout()
plt.savefig("/content/feature_importance.png", dpi=120)
plt.show()

top3 = [feat_names[i] for i in sorted_idx[:3]]
print(f"[✓] Top-3 features: {top3}")

# Save for dashboard
np.save(f"{MODEL_DIR}/shap_mean_values.npy",   importance)   # same key, dashboard reads this
np.save(f"{MODEL_DIR}/shap_feature_names.npy", np.array(feat_names))

# ── 9. Save model ─────────────────────────────────────────────────────────────
model.save(f"{MODEL_DIR}/best_model.keras")
model.save(f"{MODEL_DIR}/best_model.h5")
print(f"\n[✓] Model saved → {MODEL_DIR}/")

# ── 10. Smoke test ────────────────────────────────────────────────────────────
print("\n[→] Smoke test …")
for i, sample in enumerate(X_te[:4]):
    p   = float(model.predict(sample[np.newaxis], verbose=0)[0][0])
    h   = 1.0 - p
    rul = h * 168.0
    lvl = "OK" if h > 0.65 else ("WARNING" if h > 0.35 else "CRITICAL")
    print(f"  Sample {i+1}: Health={h:.2f}  RUL={rul:.0f}h  [{lvl}]")

# ── 11. Download reminder ─────────────────────────────────────────────────────
print("\n" + "="*55)
print("✅  Step 2 complete!")
print("\n  ⬇  Run this to download your model:")
print("""
  from google.colab import files
  import shutil
  shutil.make_archive('/content/sugarsense_model', 'zip',
                      '/content/sugarsense_model')
  files.download('/content/sugarsense_model.zip')
""")
print("="*55)

In [ ]:
!pip install streamlit plotly tensorflow scikit-learn scipy numpy


physics demo

In [ ]:
"""
SugarSense — Step 3: Sugar-Specific Physics Simulation
=======================================================
Connects bearing health → juice extraction drop → Brix deviation →
sugar recovery rate → revenue loss + FRP farmer risk.

Run on Colab (CPU is fine) after step2, or standalone.
Also imported directly by step4_dashboard.py — all physics lives here.

Physics chain:
  Bearing health → roller pressure drop → juice extraction loss
  → Brix deviation → evaporator ODE → recovery rate
  → ₹ revenue loss + FRP payment risk
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import odeint
import json
import os

OUTPUT_DIR = "/content/sugarsense_simulation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# MILL BASELINE  (all numbers sourced — see README)
# ═══════════════════════════════════════════════════════════════════════════════

class MillBaseline:
    """
    Mutable class (not dataclass) so Streamlit sidebar sliders can update values.
    """
    def __init__(self):
        self.tcd                    = 2500.0      # Tonnes Cane per Day
        self.baseline_recovery      = 10.5        # % recovery (industry benchmark)
        self.msp_per_quintal        = 3600.0      # ₹/quintal — Govt MSP 2024-25
        self.tonnes_per_hour        = 2500.0/24   # ~104.2
        self.revenue_per_day        = 9_450_000.0 # ₹ (calculated)
        self.downtime_cost_per_hour = 700_000.0   # ₹70L/hr — ABB India Survey 2023
        self.frp_per_tonne_cane     = 340.0       # ₹/tonne — FRP 2024-25

MILL = MillBaseline()


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 2A — Bearing Health → Juice Extraction Efficiency
# ═══════════════════════════════════════════════════════════════════════════════

def bearing_to_extraction(health: float) -> float:
    """
    Nonlinear mapping: bearing health (0–1) → juice extraction efficiency (0–1).

    Physics:
    - Sugar mill rollers run at ~5 RPM under massive compressive load.
    - Bearing degradation → increased friction → roller gap widens →
      lower pressing pressure → less juice extracted per tonne of cane.
    - Effect is nonlinear: stable above health 0.6, accelerates below 0.4.
    """
    health = float(np.clip(health, 0.0, 1.0))
    if health >= 0.80: return 1.00
    if health >= 0.50: return 1.00 - 0.08 * (0.80 - health) / 0.30
    if health >= 0.20: return 0.92 - 0.20 * (0.50 - health) / 0.30
    return float(max(0.72 - 0.60 * (0.20 - health) / 0.20, 0.05))


def juice_brix(extraction_eff: float, nominal_brix: float = 16.5) -> float:
    """
    Mixed-juice Brix as function of extraction efficiency.

    When pressing pressure drops → under-extracted cane → diluted juice.
    Each 1% drop in extraction below 95% → ~0.28°Bx drop in mixed juice.
    Nominal Indian cane juice Brix: 16–18°Bx.
    """
    loss_pct  = (1.0 - extraction_eff) * 100.0
    brix_drop = max(0.0, loss_pct - 5.0) * 0.28
    return float(max(nominal_brix - brix_drop, 8.0))


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 2B — Falling-Film Evaporator ODE (5-effect, standard Indian mills)
# ═══════════════════════════════════════════════════════════════════════════════

def evaporator_odes(state, t, feed_rate, inlet_brix):
    """
    State: Brix in each of 5 evaporator effects [°Bx].
    Each effect drives toward a target Brix proportional to feed rate.
    tau = 0.5 hr residence time (industry standard).
    """
    tau          = 0.5
    target_brix  = [25.0, 35.0, 45.0, 55.0, 65.0]
    feed_factor  = feed_rate / MILL.tonnes_per_hour
    return [(target_brix[i] * feed_factor - state[i]) / tau for i in range(5)]


def run_evaporator(health: float, sim_hours: float = 12.0) -> dict:
    """
    Full evaporator simulation for a bearing health score.

    Returns:
        recovery_rate (%)   — key output for economic layer
        final_brix_syrup    — °Bx leaving last effect
        inlet_brix          — °Bx of juice entering evaporator
        extraction_eff      — juice extraction efficiency
        crystallisation_risk — True if syrup Brix < 58°Bx
    """
    extr      = bearing_to_extraction(health)
    brix_in   = juice_brix(extr)
    feed_rate = MILL.tonnes_per_hour * extr

    y0  = [24.0, 34.0, 44.0, 54.0, 64.0]
    t   = np.linspace(0, sim_hours, 200)
    sol = odeint(evaporator_odes, y0, t, args=(feed_rate, brix_in))

    final_brix = float(sol[-1, 4])
    purity     = 0.87 + 0.03 * min(brix_in / 16.5, 1.0)
    recovery   = float(np.clip((brix_in / 100.0) * extr * purity * 100.0, 5.0, 12.0))

    return {
        "extraction_eff"       : round(extr, 4),
        "inlet_brix"           : round(brix_in, 2),
        "final_brix_syrup"     : round(final_brix, 2),
        "recovery_rate"        : round(recovery, 3),
        "feed_rate_tph"        : round(feed_rate, 2),
        "crystallisation_risk" : bool(final_brix < 58.0),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 3 — Economic Impact Engine
# ═══════════════════════════════════════════════════════════════════════════════

def compute_economic_impact(health: float, rul_hours: float) -> dict:
    """
    Full pipeline: health score → process sim → ₹ impact.

    Returns a flat dict used by both the CLI demo and the Streamlit dashboard.
    """
    health    = float(np.clip(health, 0.0, 1.0))
    rul_hours = float(max(rul_hours, 0.0))

    sim = run_evaporator(health)

    recovery      = sim["recovery_rate"]
    rec_drop_pct  = (MILL.baseline_recovery - recovery) / MILL.baseline_recovery

    sugar_nom  = MILL.tcd * MILL.baseline_recovery / 100.0
    sugar_cur  = MILL.tcd * recovery / 100.0
    rev_nom    = sugar_nom * MILL.msp_per_quintal * 10.0   # ₹ (quintal = 100 kg)
    rev_cur    = sugar_cur * MILL.msp_per_quintal * 10.0
    daily_loss = rev_nom - rev_cur

    # P(failure within 48h) — sigmoid centred at health=0.35
    p_fail        = float(1.0 / (1.0 + np.exp(5.0 * (health - 0.35))))
    exp_down_hrs  = p_fail * 12.0                           # expected emergency downtime
    loss_48h      = daily_loss * 2.0 + exp_down_hrs * MILL.downtime_cost_per_hour

    # FRP farmer payment risk
    frp_gap       = (MILL.tcd * rec_drop_pct) * MILL.frp_per_tonne_cane  # ₹/day shortfall
    frp_risk_days = max(0.0, frp_gap / 1_000_000.0)        # normalised to 10L threshold

    # Maintenance ROI (bearing replacement ~₹1.5L)
    maint_cost = 150_000.0
    roi        = (loss_48h - maint_cost) / maint_cost

    # Alert & recommendation
    if health > 0.65:
        urgency = "OK"
        action  = f"✓ Healthy. Inspect in {max(1, int(rul_hours/24))} days."
    elif health > 0.35:
        urgency = "WARNING"
        action  = (f"⚠ Replace within {int(rul_hours)} hrs. "
                   f"Save ₹{loss_48h/100_000:.1f}L by acting now.")
    else:
        urgency = "CRITICAL"
        action  = (f"🚨 Replace IMMEDIATELY. Failure risk: {p_fail*100:.0f}%. "
                   f"Every hour costs ₹{MILL.downtime_cost_per_hour/100_000:.0f}L.")

    return {
        # identity
        "health"            : round(health, 3),
        "rul_hours"         : round(rul_hours, 1),
        # process
        "recovery"          : round(recovery, 3),
        "recovery_drop_pct" : round(rec_drop_pct * 100, 2),
        "brix_in"           : sim["inlet_brix"],
        "final_brix"        : sim["final_brix_syrup"],
        "extr"              : sim["extraction_eff"],
        "cryst_risk"        : sim["crystallisation_risk"],
        # economic
        "rev_nom"           : round(rev_nom),
        "rev_cur"           : round(rev_cur),
        "daily_loss"        : round(daily_loss),
        "loss_48h"          : round(loss_48h),
        "frp_days"          : round(frp_risk_days, 2),
        "p_fail"            : round(p_fail, 3),
        "roi_x"             : round(roi, 1),
        # recommendation
        "urgency"           : urgency,
        "action"            : action,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# CLI DEMO — 4 bearings matching the dashboard mockup
# ═══════════════════════════════════════════════════════════════════════════════

DEMO_BEARINGS = [
    {"name": "Roller Mill #1", "health": 0.82, "rul": 97.0},
    {"name": "Roller Mill #2", "health": 0.34, "rul": 31.0},   # CRITICAL
    {"name": "Centrifugal #1", "health": 0.61, "rul": 58.0},
    {"name": "Conveyor #3",    "health": 0.91, "rul": 142.0},
]


def run_demo():
    results = []
    for b in DEMO_BEARINGS:
        r = compute_economic_impact(b["health"], b["rul"])
        r["name"] = b["name"]
        results.append(r)
        print(f"\n{'='*62}")
        print(f"  {b['name']:20s} | Health: {r['health']:.0%} | RUL: {r['rul_hours']:.0f} hrs")
        print(f"  Recovery      : {r['recovery']:.2f}% (nominal: {MILL.baseline_recovery}%)")
        print(f"  Juice Brix    : {r['brix_in']:.1f}°Bx | Cryst risk: {r['cryst_risk']}")
        print(f"  Daily revenue : ₹{r['rev_cur']/1e5:.1f}L (vs nominal ₹{r['rev_nom']/1e5:.1f}L)")
        print(f"  Loss if delay : ₹{r['loss_48h']/1e5:.1f}L | FRP risk: {r['frp_days']:.1f} days")
        print(f"  [{r['urgency']}] {r['action']}")

    with open(f"{OUTPUT_DIR}/simulation_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n[✓] Results → {OUTPUT_DIR}/simulation_results.json")
    return results


# ═══════════════════════════════════════════════════════════════════════════════
# VISUALISATION — sweep bearing health 0 → 100%
# ═══════════════════════════════════════════════════════════════════════════════

def plot_health_sweep():
    hs     = np.linspace(0.05, 1.0, 100)
    extrs  = [bearing_to_extraction(h) for h in hs]
    brixs  = [juice_brix(e) for e in extrs]
    recs   = [run_evaporator(h)["recovery_rate"] for h in hs]
    losses = [compute_economic_impact(h, h*168)["loss_48h"] / 1e5 for h in hs]

    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor("#0d1117")
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

    cfg = [
        (0, 0, extrs,  "Juice Extraction Efficiency",            "Efficiency",  "#00b4d8", 0.95,  "Nominal 0.95"),
        (0, 1, recs,   "Sugar Recovery Rate (%)",                "Recovery %",  "#06d6a0", 10.5,  "Critical 10.5%"),
        (1, 0, brixs,  "Juice Brix (°Bx)",                      "Brix (°Bx)",  "#ffd166", 15.0,  "Min 15°Bx"),
        (1, 1, losses, "Revenue at Risk — delay 48h (₹ Lakh)",   "₹ Lakh",     "#ef233c", None,  None),
    ]

    for r, c, ydata, title, ylabel, color, thresh, tlabel in cfg:
        ax = fig.add_subplot(gs[r, c])
        ax.set_facecolor("#161b22")
        ax.plot(hs * 100, ydata, color=color, linewidth=2.5)
        ax.fill_between(hs * 100, ydata, alpha=0.12, color=color)
        ax.axvspan(0,  35, alpha=0.09, color="#ef233c")
        ax.axvspan(35, 65, alpha=0.05, color="#ffd166")
        if thresh is not None:
            ax.axhline(thresh, color="white", ls="--", lw=1, alpha=0.6, label=tlabel)
            ax.legend(fontsize=8, facecolor="#0d1117", labelcolor="white")
        ax.set_title(title, color="white", fontsize=10, pad=8)
        ax.set_xlabel("Bearing Health (%)", color="#adb5bd", fontsize=9)
        ax.set_ylabel(ylabel, color="#adb5bd", fontsize=9)
        ax.tick_params(colors="#adb5bd")
        for spine in ax.spines.values():
            spine.set_color("#30363d")
        ax.grid(True, alpha=0.2, color="#30363d")

    fig.suptitle(
        "SugarSense — Bearing Health Impact Chain\n"
        "Degradation → extraction loss → recovery drop → revenue at risk",
        color="white", fontsize=13, fontweight="bold", y=0.99
    )
    out = f"{OUTPUT_DIR}/health_sweep.png"
    plt.savefig(out, dpi=150, facecolor="#0d1117", bbox_inches="tight")
    plt.show()
    print(f"[✓] Health sweep → {out}")


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("SugarSense — Physics & Economic Simulation")
    print("=" * 62)
    run_demo()
    print("\n[→] Generating visualisation …")
    plot_health_sweep()
    print("\n✅  Step 3 complete. Run step4_dashboard.py on your local machine next.")

dashboard

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "gradio", "plotly", "scipy", "--quiet"], check=True)
print("Done")

In [ ]:
"""
SugarSense — Step 4: Gradio Dashboard (Colab)
=============================================
Run this cell. Gradio gives you a public URL automatically — no token needed.
"""

import numpy as np
import gradio as gr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import odeint
import pickle, os, warnings
warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════════
# PHYSICS ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

class Mill:
    tcd                    = 2500.0
    baseline_recovery      = 10.5
    msp_per_quintal        = 3600.0
    tonnes_per_hour        = 2500.0 / 24
    downtime_cost_per_hour = 700_000.0
    frp_per_tonne_cane     = 340.0

MILL = Mill()

def bearing_to_extraction(h):
    h = float(np.clip(h, 0, 1))
    if h >= 0.80: return 1.00
    if h >= 0.50: return 1.00 - 0.08 * (0.80 - h) / 0.30
    if h >= 0.20: return 0.92 - 0.20 * (0.50 - h) / 0.30
    return float(max(0.72 - 0.60 * (0.20 - h) / 0.20, 0.05))

def juice_brix(extr, nom=16.5):
    return float(max(nom - max(0.0, (1 - extr) * 100 - 5.0) * 0.28, 8.0))

def evap_odes(state, t, feed_rate):
    tau = 0.5
    targets = [25.0, 35.0, 45.0, 55.0, 65.0]
    ff = feed_rate / MILL.tonnes_per_hour
    return [(targets[i] * ff - state[i]) / tau for i in range(5)]

def compute_impact(health, rul_hours, tcd=2500, msp=3600):
    MILL.tcd             = float(tcd)
    MILL.msp_per_quintal = float(msp)
    MILL.tonnes_per_hour = MILL.tcd / 24.0

    health    = float(np.clip(health, 0, 1))
    rul_hours = float(max(rul_hours, 0))

    extr      = bearing_to_extraction(health)
    brix_in   = juice_brix(extr)
    feed      = MILL.tonnes_per_hour * extr
    sol       = odeint(evap_odes, [24,34,44,54,64], np.linspace(0,12,200), args=(feed,))
    final_brix= float(sol[-1, 4])
    purity    = 0.87 + 0.03 * min(brix_in / 16.5, 1.0)
    recovery  = float(np.clip((brix_in / 100) * extr * purity * 100, 5.0, 12.0))

    rec_drop  = (MILL.baseline_recovery - recovery) / MILL.baseline_recovery
    rev_nom   = (MILL.tcd * MILL.baseline_recovery / 100) * MILL.msp_per_quintal * 10
    rev_cur   = (MILL.tcd * recovery / 100) * MILL.msp_per_quintal * 10
    daily_loss= rev_nom - rev_cur
    p_fail    = float(1.0 / (1.0 + np.exp(5.0 * (health - 0.35))))
    loss_48h  = daily_loss * 2 + p_fail * 12 * MILL.downtime_cost_per_hour
    frp_days  = max(0.0, (MILL.tcd * rec_drop * MILL.frp_per_tonne_cane) / 1_000_000)

    if health > 0.65:
        urgency = "✅ OK"
        action  = f"Healthy. Inspect in {max(1, int(rul_hours/24))} days."
    elif health > 0.35:
        urgency = "⚠️ WARNING"
        action  = f"Replace within {int(rul_hours)} hrs — save ₹{loss_48h/1e5:.1f}L."
    else:
        urgency = "🚨 CRITICAL"
        action  = f"Replace IMMEDIATELY. Failure risk {p_fail*100:.0f}%. Every hr costs ₹{MILL.downtime_cost_per_hour/1e5:.0f}L."

    return dict(
        health=health, rul_hours=rul_hours, recovery=recovery,
        brix_in=brix_in, final_brix=final_brix,
        cryst_risk=bool(final_brix < 58),
        rev_nom=rev_nom, rev_cur=rev_cur,
        daily_loss=daily_loss, loss_48h=loss_48h,
        frp_days=round(frp_days, 2), p_fail=p_fail,
        urgency=urgency, action=action, extr=extr,
    )

# ═══════════════════════════════════════════════════════════════════════════════
# CHART BUILDERS
# ═══════════════════════════════════════════════════════════════════════════════

DARK = dict(
    paper_bgcolor="#0f1829", plot_bgcolor="#0f1829",
    font=dict(color="#e2e8f0", family="sans-serif"),
    margin=dict(l=16, r=16, t=44, b=16),
)
C = {"✅ OK": "#00e5b0", "⚠️ WARNING": "#f59e0b", "🚨 CRITICAL": "#f43f5e"}

def make_gauge_fig(impacts, bearing_names):
    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type":"indicator"},{"type":"indicator"}],
               [{"type":"indicator"},{"type":"indicator"}]],
        vertical_spacing=0.08, horizontal_spacing=0.04
    )
    for idx, (imp, name) in enumerate(zip(impacts, bearing_names)):
        row, col = divmod(idx, 2)
        c = C[imp["urgency"]]
        fig.add_trace(go.Indicator(
            mode="gauge+number",
            value=imp["health"] * 100,
            title=dict(text=name, font=dict(size=11, color="#64748b")),
            number=dict(suffix="%", font=dict(size=20, color=c)),
            gauge=dict(
                axis=dict(range=[0,100], tickwidth=1, tickcolor="#1a2540",
                          tickfont=dict(color="#334155", size=8)),
                bar=dict(color=c, thickness=0.22),
                bgcolor="#0c1829", borderwidth=1, bordercolor="#1a2540",
                steps=[dict(range=[0,35],  color="#1f0a10"),
                       dict(range=[35,65], color="#1a1305"),
                       dict(range=[65,100],color="#071510")],
                threshold=dict(line=dict(color="white", width=2),
                               thickness=0.75, value=35),
            )), row=row+1, col=col+1)
    fig.update_layout(**DARK, height=340,
                      title="Bearing Health Scores", title_font_size=13)
    return fig

def make_recovery_fig(impacts, bearing_names):
    colors = [C[i["urgency"]] for i in impacts]
    fig = go.Figure(go.Bar(
        x=bearing_names,
        y=[i["recovery"] for i in impacts],
        marker_color=colors,
        text=[f"{i['recovery']:.2f}%" for i in impacts],
        textposition="outside",
        textfont=dict(size=12)
    ))
    fig.add_hline(y=10.5, line_dash="dash", line_color="white", line_width=1.5,
                  annotation_text="⚠ Critical 10.5%",
                  annotation_font=dict(color="#94a3b8", size=11))
    fig.update_layout(**DARK, title="Sugar Recovery Rate (%)",
                      title_font_size=13, height=300,
                      yaxis=dict(range=[0,13], gridcolor="#1a2540"),
                      xaxis=dict(gridcolor="#1a2540"), showlegend=False)
    return fig

def make_revenue_fig(impacts, bearing_names):
    colors = [C[i["urgency"]] for i in impacts]
    fig = go.Figure(go.Bar(
        x=bearing_names,
        y=[i["loss_48h"] / 1e5 for i in impacts],
        marker_color=colors,
        text=[f"₹{i['loss_48h']/1e5:.1f}L" for i in impacts],
        textposition="outside",
        textfont=dict(size=12)
    ))
    fig.update_layout(**DARK, title="Revenue at Risk — 48h delay (₹ Lakh)",
                      title_font_size=13, height=300,
                      yaxis=dict(gridcolor="#1a2540"),
                      xaxis=dict(gridcolor="#1a2540"), showlegend=False)
    return fig

def make_oee_fig(imp, name):
    hrs = np.linspace(0, 72, 200)
    ht  = imp["health"] * np.exp(-hrs / max(imp["rul_hours"], 1))
    oee = [bearing_to_extraction(h) * 100 for h in ht]

    fig = go.Figure()
    fig.add_hrect(y0=0,  y1=70, fillcolor="#1f0a10", opacity=0.35, line_width=0)
    fig.add_hrect(y0=70, y1=85, fillcolor="#1a1305", opacity=0.25, line_width=0)
    fig.add_trace(go.Scatter(
        x=hrs, y=oee, mode="lines",
        line=dict(color="#f43f5e", width=2.5),
        fill="tozeroy", fillcolor="rgba(244,63,94,0.07)",
        name="OEE %"
    ))
    fig.add_vline(x=0, line_dash="dash", line_color="#00e5b0",
                  annotation_text="Now", annotation_font_color="#00e5b0")
    if imp["rul_hours"] <= 72:
        fig.add_vline(x=imp["rul_hours"], line_dash="dot", line_color="#f43f5e",
                      annotation_text="Est. Failure", annotation_font_color="#f43f5e")
    fig.update_layout(**DARK,
        title=f"OEE Degradation Timeline — {name} (next 72h)",
        title_font_size=13, height=300,
        yaxis=dict(range=[0,105], title="OEE (%)", gridcolor="#1a2540"),
        xaxis=dict(title="Hours from now", gridcolor="#1a2540"),
        showlegend=False)
    return fig

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY TEXT BUILDER
# ═══════════════════════════════════════════════════════════════════════════════

def build_summary(impacts, bearing_names):
    lines = []
    total_risk = sum(i["loss_48h"] for i in impacts) / 1e5

    lines.append(f"## ⚙ SugarSense — Live Analysis")
    lines.append(f"**Total revenue at risk (48h delay): ₹{total_risk:.1f} Lakh**\n")

    for name, imp in zip(bearing_names, impacts):
        lines.append(f"---")
        lines.append(f"### {imp['urgency']} &nbsp; {name}")
        lines.append(f"| Metric | Value |")
        lines.append(f"|--------|-------|")
        lines.append(f"| Health Score | **{imp['health']:.0%}** |")
        lines.append(f"| RUL | **{imp['rul_hours']:.0f} hrs** |")
        lines.append(f"| Recovery Rate | **{imp['recovery']:.2f}%** {'⚠ below threshold' if imp['recovery'] < 10.5 else '✓'} |")
        lines.append(f"| Juice Brix | {imp['brix_in']:.1f}°Bx |")
        lines.append(f"| Crystallisation Risk | {'⚠ Yes' if imp['cryst_risk'] else '✓ No'} |")
        lines.append(f"| Daily Revenue Loss | ₹{imp['daily_loss']/1e5:.1f}L |")
        lines.append(f"| Loss if Delay 48h | **₹{imp['loss_48h']/1e5:.1f}L** |")
        lines.append(f"| FRP Payment Risk | {imp['frp_days']:.1f} days |")
        lines.append(f"| Failure Prob (48h) | {imp['p_fail']*100:.0f}% |")
        lines.append(f"| **Action** | {imp['action']} |")

    lines.append(f"\n---")
    lines.append(f"*SugarSense · Team Muffin · Sneha Chakraborty & Divyansh Pathak · ISMA Sugar-NXT Hackathon 2026*")
    lines.append(f"*Sources: ABB India Survey 2023 · ISMA Dec 2025 · Govt MSP 2024-25*")
    return "\n".join(lines)

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN DASHBOARD FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

BEARING_NAMES = ["Roller Mill #1", "Roller Mill #2", "Centrifugal #1", "Conveyor #3"]

def dashboard(
    h1, r1, h2, r2, h3, r3, h4, r4,
    tcd, msp,
    focus_bearing
):
    healths = [h1, h2, h3, h4]
    ruls    = [r1, r2, r3, r4]
    impacts = [compute_impact(h, r, tcd, msp) for h, r in zip(healths, ruls)]

    focus_idx = BEARING_NAMES.index(focus_bearing)

    gauge_fig    = make_gauge_fig(impacts, BEARING_NAMES)
    recovery_fig = make_recovery_fig(impacts, BEARING_NAMES)
    revenue_fig  = make_revenue_fig(impacts, BEARING_NAMES)
    oee_fig      = make_oee_fig(impacts[focus_idx], focus_bearing)
    summary_md   = build_summary(impacts, BEARING_NAMES)

    return gauge_fig, recovery_fig, revenue_fig, oee_fig, summary_md

# ═══════════════════════════════════════════════════════════════════════════════
# GRADIO UI
# ═══════════════════════════════════════════════════════════════════════════════

CSS = """
body, .gradio-container { background: #080d18 !important; color: #e2e8f0 !important; }
.gr-button-primary { background: #00e5b0 !important; color: #080d18 !important; font-weight: 700 !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=CSS, title="SugarSense — AI Equipment Intelligence") as demo:

    gr.Markdown("""
# ⚙ SugarSense — AI Equipment Intelligence
### Team Muffin · Sneha Chakraborty & Divyansh Pathak · ISMA Sugar-NXT Hackathon 2026
*From a worn bearing to a farmer's delayed payment — we predict it before it happens.*
---
    """)

    # ── Controls ──────────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🏭 Mill Settings")
            tcd = gr.Slider(500,  10000, value=2500, step=100,  label="Mill Capacity (TCD)")
            msp = gr.Slider(3000, 4500,  value=3600, step=50,   label="MSP ₹/quintal")

        with gr.Column(scale=1):
            gr.Markdown("### ⚙ Roller Mill #1")
            h1 = gr.Slider(0.0, 1.0,   value=0.82, step=0.01, label="Health Score")
            r1 = gr.Slider(0.0, 200.0, value=97.0, step=1.0,  label="RUL (hours)")

        with gr.Column(scale=1):
            gr.Markdown("### ⚙ Roller Mill #2  🚨")
            h2 = gr.Slider(0.0, 1.0,   value=0.34, step=0.01, label="Health Score")
            r2 = gr.Slider(0.0, 200.0, value=31.0, step=1.0,  label="RUL (hours)")

        with gr.Column(scale=1):
            gr.Markdown("### ⚙ Centrifugal #1")
            h3 = gr.Slider(0.0, 1.0,   value=0.61, step=0.01, label="Health Score")
            r3 = gr.Slider(0.0, 200.0, value=58.0, step=1.0,  label="RUL (hours)")

        with gr.Column(scale=1):
            gr.Markdown("### ⚙ Conveyor #3")
            h4 = gr.Slider(0.0, 1.0,   value=0.91, step=0.01, label="Health Score")
            r4 = gr.Slider(0.0, 200.0, value=142.0,step=1.0,  label="RUL (hours)")

    with gr.Row():
        focus = gr.Dropdown(
            choices=BEARING_NAMES, value="Roller Mill #2",
            label="OEE Timeline — focus bearing"
        )
        run_btn = gr.Button("🔄 Update Dashboard", variant="primary", scale=2)

    gr.Markdown("---")

    # ── Outputs ───────────────────────────────────────────────────────────────
    with gr.Row():
        gauge_out = gr.Plot(label="Bearing Health Gauges")

    with gr.Row():
        recovery_out = gr.Plot(label="Sugar Recovery Rate")
        revenue_out  = gr.Plot(label="Revenue at Risk")

    with gr.Row():
        oee_out = gr.Plot(label="OEE Degradation Timeline")

    summary_out = gr.Markdown(label="Full Analysis")

    # ── Wire up ───────────────────────────────────────────────────────────────
    inputs  = [h1, r1, h2, r2, h3, r3, h4, r4, tcd, msp, focus]
    outputs = [gauge_out, recovery_out, revenue_out, oee_out, summary_out]

    run_btn.click(fn=dashboard, inputs=inputs, outputs=outputs)

    # Auto-run on load
    demo.load(fn=dashboard, inputs=inputs, outputs=outputs)

    # Live update on any slider change
    for slider in [h1, r1, h2, r2, h3, r3, h4, r4, tcd, msp, focus]:
        slider.change(fn=dashboard, inputs=inputs, outputs=outputs)

# ── Launch ────────────────────────────────────────────────────────────────────
print("\n[→] Launching SugarSense dashboard …")
demo.launch(share=True, quiet=True)
print("\n✅  Dashboard is live! Open the public URL above.")